# Hackathon - ByteCraft
---
This notebook contains the approach, design and possible solutions for the Smart India Hackathon 2026 - Problem Statement - SIH26072.

## SIH26072 - AI-ML based Nowcasting of thunderstorm and lightning using atmospheric observation including multiple radars, satellite, lightning and model data:
---
This notebook will discuss Nowcasting techniques using Machine learning and AI-models and how they can be efficient in short period forecast and weather changes without relying on large scale simulation models that can take way long to run and make predictions.

### What is **Nowcasting**?:

**Nowcasting** is a high-speed, hyper-local weather forecasting technique for immediate future - typically covering 0-6 hours.

### How is it different from standard forecasting techniques?

Standard weather forecasting techniques are for predicting the weather over a longer period like days and weeks even while Nowcasting techniques are for shorter periods to give a quick prediction of what is around the corner for the day.

> **Example Analogy:** If standard weather forecasting is like predicting where a marathon runner will be three hours from now based on their pacing and training, nowcasting is like tracking where a sprinter will step in next ten seconds by watching their balance, muscle tension and foot placement.

### Drawbacks with Traditional Weather Forecasts:

Traditional weather forecasting relies heavily on giant numerical simulations called **Numerical Weather Predictions(NWP)**. These models take observations across the entire planet, solve complex fluid dynamics and thermodynamics equations and tells us what the weather is going to be tomorrow or day after tomorrow.

While this is incredibly accurate in predicting the weather patterns on the long run - it falls short for certain sudden changes in the weather, that break predefined models - like an **anomaly** or an **edge-case**.

Thunderstroms and sudden downpours have such two characterisitcs that can break traditional models:

1. **They are short lived and localised:** A severe thunderstorm cloud can form, dump heavy rains, drop lighting and completely dissipate within 45-90 minutes across an area just 5-15 KMs wide. This is extremely difficult for a large simulation model specifically designed to predict results for a larger surface area and longer periods of time.

2. **They take too long to compute:** Gathering data, cleaning it and running large scale simulations on a global or regional supercomputer cluster takes a lot of time. By the time the forecast is calculated, the storm has hit and already vanished.

### How traditional Nowcasting works?

Instead of running slow physics simulations from scratch, traditional nowcasting operates near real time by doing these three basic operations:

1. **Where is the storm right now?**: Sensors like **Doppler radar** (which spots rain droplets and wind inside clouds) and **Satellites** (which look at cloud temperature from the space) update every 5-10 minutes.

2. **Where is it moving?**: By comparing snapshots from 10 minutes ago, 5 minutes ago and right now, algorithms calculate and storm's velocity vector and extrapolate its path forward predicting where it is going next.

3. **Is it growing, dying or about to form?**: This is the hardest part. A storm isn't moving in a straight line or have the same size (like a ball moving across a table), instead it can suddenly change direction, intensify in size, turn into a severe hailstorm or evaporate into a severe hailstorm or evaporate into thin air within 20 minutes depending on local temperature, humidity, and rising warm air.

Conventional operational nowcasting methods have been implementing three main techniques:

#### Centroid and Object Tracking:

- **Mechanism:** Algorithms identify isolated storm cells by thresholding radar reflectivity and fit them into geometric ellipsoids or polygons. **Note: Why??**

- **Tracking:** By matching centroids between sequential radar scans (every 5-10 mins) using distance minimisation or overlap, the algorithm computes a translation vector ($Δx, Δy$) and projects the polygon forward.

- **Limitation:** Treats storms as rigid objects. It struggles when cells undergo splitting, merging, or rapid spatial deformation.

#### Optical Flow and Eulerian-Lagrangian Advection:

- **Mechanisms:** Instead of tracking segmented objects, optical flow algorithms estimate a dense, 2D continous motion field ($u, v$) across every pixel of consecutive radar or satellite grids.

- **Extrapolation:** The current reflectivity field $\textit{I}(x,y,t)$ is advected forward using a semi-Lagrangian scheme along the calculated velocity vectors:

$$
\dfrac{dI}{dt} ≈ 0 \implies I(x + uΔt, y + vΔt, t + Δt) ≈ I(x,y,t)
$$

- **Stochastic Extensions:** Systems like **STEPS** decompose precipitation into spatial cascade levels (filtering by spatial frequency) and add autoregressive noise to mimic predictability decay over time.

- **Limiatations:** Pure advection assumes precipitation intensity is conserved along the motion trajectory. It is blind to **Convective Initiation(CI)** (birth of new cells) and **dissipation** (cell decay).

#### Rapid Update Convective-Allowing Models (CAMs):

**Mechanism:** Running localised, non-hydrostatic numerical weather prediction (NWP) models (e.g: HRRR, localised WRF) with grid spacings of 1-3 KMs and frequent data assimilation cycles (hourly or sub-hourly).

**Limitations:**
1. **Latency:** Data assimilation and running differential equations still take a long time.

2. **Spin-up Error:** Newly ingested radar echoes can take 15-30 simulation minutes to balance with model microphysics and thermodynamics, during which precipitation forecasts are unreliable.

### Nowcasting with AI and Machine/Deep Learning Methods:

Machine/Deep Learning models can modernise by moving beyong simple kinematic motion tracking to learning the non-linear, multi-scale physical evolution of convective systems directly from multi-modal sensor streams.

#### **Traditional Nowcating Flow:**

<img src="flow1.png" alt="Traditional Nowcasting Pipeline" width="1000">

#### **Modern AI Flow:**

<img src="flow2.png" alt="Traditional Nowcasting Pipeline" width="1000">

#### Objectives of Nowcasting using ML/DL Methods:

1. Fast and Accurate: 
2. Scalability:

In [1]:
import torch
import torch.nn as nn
import numpy as np

# ---------------------------------------------------------
# 1. SYNTHETIC RADAR DATA GENERATOR
# ---------------------------------------------------------
def generate_storm_sequence(batch_size=4, total_seq=6, size=64):
    """
    Generates synthetic 'radar' sequences of a moving storm cell.
    Returns tensor of shape: (batch_size, total_seq, channels(1), width, height)
    """
    data = np.zeros((batch_size, total_seq, 1, size, size), dtype=np.float32)
    
    for b in range(batch_size):
        # Spawn a storm at a random starting coordinate
        x, y = np.random.randint(10, 30), np.random.randint(10, 30)
        
        # Give the storm a random velocity (wind vector)
        vx, vy = np.random.uniform(1, 3), np.random.uniform(1, 3)
        
        for t in range(total_seq):
            # Create a 2D coordinate grid
            X, Y = np.meshgrid(np.arange(size), np.arange(size))
            
            # Draw the storm as a Gaussian blob (heat map)
            blob = np.exp(-(((X - x)**2 + (Y - y)**2) / 20.0))
            data[b, t, 0, :, :] = blob
            
            # Advect (move) the storm forward in time
            x += vx
            y += vy
            
    return torch.tensor(data)

# ---------------------------------------------------------
# 2. THE MODEL: CONVLSTM CELL
# ---------------------------------------------------------
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size):
        super().__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size // 2
        
        # One large convolution handles all four LSTM gates simultaneously
        self.conv = nn.Conv2d(in_channels=input_dim + hidden_dim,
                              out_channels=4 * hidden_dim,
                              kernel_size=kernel_size, 
                              padding=padding)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        
        # Stack the new radar frame with the hidden memory state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)
        
        # Split the tensor into the 4 LSTM gates
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        
        i = torch.sigmoid(cc_i)      # Input gate: What new data to keep?
        f = torch.sigmoid(cc_f)      # Forget gate: What old memory to drop?
        o = torch.sigmoid(cc_o)      # Output gate: What to show next?
        g = torch.tanh(cc_g)         # Cell update: New candidate memory
        
        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        
        return h_next, c_next

# ---------------------------------------------------------
# 3. THE NOWCASTER & TRAINING LOOP
# ---------------------------------------------------------
class SimpleNowcaster(nn.Module):
    def __init__(self):
        super().__init__()
        # 1 input channel (radar), 16 hidden channels (memory), 3x3 kernel
        self.cell = ConvLSTMCell(input_dim=1, hidden_dim=16, kernel_size=3)
        # Collapse the 16 hidden channels back down to 1 grayscale radar image
        self.conv_out = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        b, seq, _, h, w = x.size()
        
        # Initialize blank memory states
        h_t = torch.zeros(b, 16, h, w).to(x.device)
        c_t = torch.zeros(b, 16, h, w).to(x.device)
        
        # Read the past radar sequence frame by frame
        for t in range(seq):
            h_t, c_t = self.cell(x[:, t, :, :, :], (h_t, c_t))
            
        # Generate the prediction for the future frame
        prediction = self.conv_out(h_t)
        return torch.sigmoid(prediction)

if __name__ == "__main__":
    print("1. Generating synthetic storm dataset...")
    # Generate 6 frames total: 5 for past context, 1 for the future target
    dataset = generate_storm_sequence(batch_size=16, total_seq=6, size=64)
    past_radar = dataset[:, :5]   # Input: Frames 0 to 4
    future_radar = dataset[:, 5]  # Target: Frame 5

    model = SimpleNowcaster()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()

    print("2. Training ConvLSTM to predict movement...")
    for epoch in range(1, 101):
        optimizer.zero_grad()
        
        # The model guesses where the storm will be
        pred = model(past_radar)
        
        # We penalize it based on how far off the guess was from reality
        loss = criterion(pred, future_radar)
        loss.backward()
        optimizer.step()
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch:03d} | Mean Squared Error Loss: {loss.item():.4f}")
            
    print("\nTraining Complete! The network has learned how to track and advect spatial objects.")

1. Generating synthetic storm dataset...
2. Training ConvLSTM to predict movement...
Epoch 020 | Mean Squared Error Loss: 0.0077
Epoch 040 | Mean Squared Error Loss: 0.0074
Epoch 060 | Mean Squared Error Loss: 0.0074
Epoch 080 | Mean Squared Error Loss: 0.0074
Epoch 100 | Mean Squared Error Loss: 0.0074

Training Complete! The network has learned how to track and advect spatial objects.
